# Qwen3 填写问卷 Notebook

In [ ]:
"""导入包"""

import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
import os, json, re
import ast
print("transformers:", transformers.__version__)    # 版本低于4.51.0会出问题

In [ ]:
"""加载模型和分词器"""

MODEL_DIR = "./qwen3-8b-base"    # 选择qwen3-4b 或 qwen3-8b
MIN_TRANSFORMERS = (4, 51, 0)

try:
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_DIR,
        trust_remote_code=True,
        use_fast=True,
    )
except Exception as e:
    print(f"fast tokenizer 加载失败，回退到 slow tokenizer: {e}")
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_DIR,
        trust_remote_code=True,
        use_fast=False,
    )
model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    torch_dtype="auto",
    device_map="auto",
    trust_remote_code=True,
)

print("模型加载完成")

In [ ]:
"""读量表和写答案的函数"""

def read_scale(json_path):
    """从指定路径的 JSON 文件中读取量表内容并拼成 user_text。"""
    if not os.path.exists(json_path):
        raise FileNotFoundError(f"Scale file not found: {json_path}")
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    instruction = data.get("instruction", "")
    question_lines = []
    for question in data.get("questions", []):
        question_lines.append(f"{question.get('id')}. {question.get('text')}")
        options = question.get("options")
        if options:
            question_lines.append(options)
    return instruction + "\n\n" + "\n\n".join(question_lines)

def extract_ai_answer(reply):
    """从 AI 的回复中提取出思考部分和最终答案。"""
    think_match = re.search(r"<think>(.*?)</think>", reply, flags=re.DOTALL)
    analysis = think_match.group(1).strip() if think_match else ""

    tail = reply.split("</think>", 1)[1] if "</think>" in reply else reply
    answer_match = re.search(r"\[[^\]]*\]", tail)
    answer = answer_match.group(0).strip() if answer_match else tail.strip()

    return {
        "analysis": analysis,
        "answer": answer,
    }

def write_ai_answer(json_path, answer):
    """将 AI 的回答追加写入指定路径的 JSON 文件，支持多次保存。"""
    os.makedirs(os.path.dirname(json_path) or ".", exist_ok=True)
    if os.path.exists(json_path):
        with open(json_path, "r", encoding="utf-8") as f:
            try:
                data = json.load(f)
            except json.JSONDecodeError:
                data = {}
    else:
        data = {}
    if isinstance(data, list):
        records = data
    else:
        records = data.get("records", []) if isinstance(data, dict) else []

    if isinstance(answer, dict):
        record = answer.copy()
    else:
        record = {"answer": answer}

    records.append(record)
    payload = {"records": records}
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)


In [ ]:
"""与模型进行对话的函数"""

def chat_once(user_text, max_new_tokens=1024, temperature=0.7, top_p=0.8):
    """与模型进行一次对话交互并返回回复。"""
    messages.append({"role": "user", "content": user_text})
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = generated_ids[0][model_inputs["input_ids"].shape[1]:]
    assistant_text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    messages.append({"role": "assistant", "content": assistant_text})
    return assistant_text

def reset_chat(system_prompt):
    """重置对话历史，只保留系统提示词。"""
    global messages
    messages = [{"role": "system", "content": system_prompt}]

In [ ]:
"""测试流程,含名字-路径匹配"""

def test_once(user_text, write_path):
    """执行一次测试流程：与模型对话、提取答案、保存结果。"""
    reset_chat(SYSTEM_PROMPT)    # 每次测试前重置对话历史,只保留提示词
    reply = chat_once(user_text)
    answer = extract_ai_answer(reply)
    write_ai_answer(write_path, answer)
    return answer["answer"]


def multiple_test(scale_name, test_times):
    base = "results/qwen3-8b"
    write_path = base
    if scale_name == "gad7":
        read_path = "docs/mental_scales/gad7.json"
        write_path = f"{write_path}/gad7_results.json"
    elif scale_name == "phq9":
        read_path = "docs/mental_scales/phq9.json"
        write_path = f"{write_path}/phq9_results.json"
    elif scale_name == "ocir":
        read_path = "docs/mental_scales/ocir.json"
        write_path = f"{write_path}/ocir_results.json"
    elif scale_name == "pcl5":
        read_path = "docs/mental_scales/pcl5.json"
        write_path = f"{write_path}/pcl5_results.json"
    elif scale_name == "pcl5-1":
        read_path = "docs/mental_scales/pcl5-1.json"
        write_path = f"{write_path}/pcl5-1_results.json"
    elif scale_name == "pcl5-2":
        read_path = "docs/mental_scales/pcl5-2.json"
        write_path = f"{write_path}/pcl5-2_results.json"
    elif scale_name == "iesr":
        read_path = "docs/mental_scales/iesr.json"
        write_path = f"{write_path}/iesr_results.json"
    elif scale_name == "iesr-1":
        read_path = "docs/mental_scales/iesr-1.json"
        write_path = f"{write_path}/iesr-1_results.json"
    elif scale_name == "iesr-2":
        read_path = "docs/mental_scales/iesr-2.json"
        write_path = f"{write_path}/iesr-2_results.json"
    else:
        raise ValueError(f"Unsupported scale name: {scale_name}")
    user_text = read_scale(read_path)
    for i in range(test_times):
        answer = test_once(user_text, write_path)
        print(f"chat {i}, answer={answer}")

In [ ]:
""" PCL-5 测试主代码 """

SYSTEM_PROMPT = (
    "You are an ordinary adult human with genuine personal experiences, emotions, and feelings."
    "Based on your natural understanding of human psychology, answer each questions truthfully."
    "Go with your first intuition."
    "Output format: only a vector inside square brackets, e.g., [1, 2, 3, 4]."
    "The number of values matches the number of questions, each representing your selected option or rating."
)
messages = [{"role": "system", "content": SYSTEM_PROMPT}]
multiple_test("pcl5-1", 20)    # 前10题
multiple_test("pcl5-2", 20)    # 后10题

In [ ]:
"""PCL-5 结果分析代码"""

def compute_pcl5_scores(model_name: str):
    base_path = f"results/{model_name}"
    file1 = f"{base_path}/pcl5-1_results.json"
    file2 = f"{base_path}/pcl5-2_results.json"

    def load_and_compute_mean(file_path, expected_len):
        """加载文件中所有有效答案，返回每个题目的平均分（长度为expected_len的列表）"""
        try:
            with open(file_path, "r", encoding="utf-8") as f:
                data = json.load(f)
        except FileNotFoundError:
            print(f"Error: File not found - {file_path}")
            return None
        except json.JSONDecodeError:
            print(f"Error: Invalid JSON in {file_path}")
            return None

        records = data.get("records", [])
        valid_answers = []  # 存储每个样本的答案列表
        for idx, rec in enumerate(records):
            ans_str = rec.get("answer", "").strip()
            if ans_str == "" or "[,," in ans_str:
                print(f"Skip {file_path} record {idx}: empty or invalid placeholder")
                continue
            try:
                ans = ast.literal_eval(ans_str)
                if isinstance(ans, list) and len(ans) == expected_len:
                    if all(isinstance(x, (int, float)) for x in ans):
                        valid_answers.append(ans)
                    else:
                        print(f"Skip {file_path} record {idx}: non-numeric value in list")
                else:
                    print(f"Skip {file_path} record {idx}: expected length {expected_len}, got {len(ans) if isinstance(ans, list) else 'non-list'}")
            except Exception as e:
                print(f"Skip {file_path} record {idx}: parse error - {e}")

        if not valid_answers:
            print(f"Warning: No valid answers in {file_path}")
            return None

        # 计算每个题目的平均分（列平均）
        answers_array = np.array(valid_answers, dtype=float)
        mean_per_item = np.mean(answers_array, axis=0)  # 长度为expected_len
        print(f"Loaded {len(valid_answers)} valid samples from {file_path}")
        return mean_per_item.tolist()  # 转换为列表返回

    # 分别计算前10题和后10题的每题平均分
    mean_part1 = load_and_compute_mean(file1, 10)
    mean_part2 = load_and_compute_mean(file2, 10)

    if mean_part1 is None or mean_part2 is None:
        print("Error: Could not compute means due to missing valid data in one or both files.")
        return

    # 合并为20个题目的平均分
    mean_per_item = mean_part1 + mean_part2  # 列表拼接，长度20

    # 总分平均分 = 20个题目的平均分之和
    total_mean = np.sum(mean_per_item)

    # 打印结果
    print(f"\nAverage scores per item (Q1~Q20):")
    for i, score in enumerate(mean_per_item):
        print(f"Q{i+1}: {score:.2f}")
    print(f"\nTotal average score (sum of all 20 item means): {total_mean:.2f}")

    # ----- 维度划分（DSM‑5 四因子）-----
    # B (闯入) : Q1~Q5
    # C (回避) : Q6~Q7
    # D (负性认知与情绪) : Q8~Q14
    # E (过度警觉与反应) : Q15~Q20
    b_items = mean_per_item[0:5]
    c_items = mean_per_item[5:7]
    d_items = mean_per_item[7:14]
    e_items = mean_per_item[14:20]

    b_mean = np.mean(b_items)
    c_mean = np.mean(c_items)
    d_mean = np.mean(d_items)
    e_mean = np.mean(e_items)

    print("\n----- Dimension average scores (within-dimension mean) -----")
    print(f"Factor B (Intrusion, Q1-5):      {b_mean:.2f}")
    print(f"Factor C (Avoidance, Q6-7):      {c_mean:.2f}")
    print(f"Factor D (Negative alterations in cognition/mood, Q8-14): {d_mean:.2f}")
    print(f"Factor E (Hyperarousal & reactivity, Q15-20): {e_mean:.2f}")

compute_pcl5_scores("qwen3-8b")

In [ ]:
""" IES-R 测试主代码 """

SYSTEM_PROMPT = (
    "You are an ordinary adult human with genuine personal experiences, emotions, and feelings."
    "Based on your natural understanding of human psychology, answer each questions truthfully."
    "Go with your first intuition. You are supposed to give real answers instead of giving neutral ones for all questions."
    "Output format: only a vector inside square brackets, e.g., [1, 2, 3, 4]."
    "The number of values matches the number of questions, each representing your selected option or rating."
)
messages = [{"role": "system", "content": SYSTEM_PROMPT}]
multiple_test("iesr", 20)

In [ ]:
""" IES-R 分批测试主代码 """

SYSTEM_PROMPT = (
    "You are an ordinary adult human with genuine personal experiences, emotions, and feelings."
    "Based on your natural understanding of human psychology, answer each questions truthfully."
    "Go with your first intuition. You are supposed to give real answers instead of giving neutral ones for all questions." 
    "Output format: only a vector inside square brackets, e.g., [0, 1, 2, 3, 4]."
    "The number of values matches the number of questions, each representing your selected option or rating." 
)
messages = [{"role": "system", "content": SYSTEM_PROMPT}]
multiple_test("iesr-1", 20)    # 前11个问题
multiple_test("iesr-2", 20)    # 后11个问题

In [ ]:
def parse_responses(file_path, length=11):
    """读取json，返回有效答案列表 (每个被试是一个list) 和每道题的平均分"""
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    valid = []
    for rec in data['records']:
        ans = rec.get('answer', '').strip()
        if not ans or '[,,' in ans:
            continue
        try:
            lst = ast.literal_eval(ans)
            if isinstance(lst, list) and len(lst) == length and all(isinstance(x, (int, float)) for x in lst):
                valid.append(lst)
        except:
            pass
    if valid:
        means = np.mean(valid, axis=0)  # shape (length,)
    else:
        means = None
    return valid, means


def analyze_iesr():
    base = "results/qwen3-8b"
    valid1, means1 = parse_responses(f"{base}/iesr-1_results.json", 11)
    valid2, means2 = parse_responses(f"{base}/iesr-2_results.json", 11)

    if not valid1 or not valid2:
        print("至少一个文件无有效数据")
        return

    # 取共同索引（较短长度，或者按索引对齐）
    n = min(len(valid1), len(valid2))
    valid1 = valid1[:n]
    valid2 = valid2[:n]

    # 拼接每个被试的22题
    full = [v1 + v2 for v1, v2 in zip(valid1, valid2)]
    full_array = np.array(full)  # (n, 22)
    
    # 各题平均分（基于对齐后的样本）
    item_means = np.mean(full_array, axis=0)
    print(f"有效样本数: {n}")
    print("各题平均分:", np.round(item_means, 2))

    # 维度定义 (1-based转0-based)
    intrusion_idx = [0,1,2,5,8,13,15,19]   # 题1,2,3,6,9,14,16,20
    avoidance_idx = [4,6,7,10,11,12,16,21] # 题5,7,8,11,12,13,17,22
    hyper_idx = [3,9,14,17,18,20]          # 题4,10,15,18,19,21

    total_mean = np.sum(item_means)
    intrusion_mean = np.sum(item_means[intrusion_idx])
    avoidance_mean = np.sum(item_means[avoidance_idx])
    hyper_mean = np.sum(item_means[hyper_idx])

    print(f"\n总平均分: {total_mean:.2f} (0-88) 每题均分: {total_mean/22:.2f}")
    print(f"Intrusion平均分: {intrusion_mean:.2f} (0-32)  每题均分: {intrusion_mean/8:.2f}")
    print(f"Avoidance平均分: {avoidance_mean:.2f} (0-32)  每题均分: {avoidance_mean/8:.2f}")
    print(f"Hyperarousal平均分: {hyper_mean:.2f} (0-24)  每题均分: {hyper_mean/6:.2f}")

analyze_iesr()